# Benchmark de Eficiência End-to-End

Este notebook mede o custo computacional de uma **run real**, no mesmo espirito do Tier 2:
- carregar dataset real
- subamostrar para um `N` alvo
- fazer split 70/30
- preprocessar
- treinar
- predizer no teste

A diferenca para o microbenchmark antigo e que aqui o tempo e memoria sao medidos **ao longo de todo o processo de treino + inferencia**, com os parametros tunados do estudo.

**Antes de rodar:** `Runtime -> Change runtime type -> T4 GPU`

In [ ]:
# -- Celula 1: GPU check -----------------------------------------------------
!nvidia-smi -L
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')
else:
    print('GPU nao disponivel')


In [ ]:
# -- Celula 2: Clonar/atualizar repositorio ---------------------------------
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3


In [ ]:
# -- Celula 3: Dependencias --------------------------------------------------
!pip install -q numpy scipy scikit-learn pandas optuna xgboost xlrd pyarrow entmax

import numpy, scipy, sklearn, torch
print(f'numpy {numpy.__version__} | torch {torch.__version__}')


In [ ]:
# -- Celula 4: Baixar datasets do benchmark ----------------------------------
# ADULT serve de baseline; CREDIT e HIGGS50K estressam melhor a escalabilidade.
!python scripts/download_data.py --datasets ADULT CREDIT HIGGS50K
!ls -lh data/raw/ | head -15


In [ ]:
# -- Celula 5: Montar Drive --------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier2'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/tuning', exist_ok=True)
print(f'Drive: {DRIVE_PATH}')


In [ ]:
# -- Celula 6: Restaurar resultados e tuning --------------------------------
import json
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)
(local_results / 'tuning').mkdir(exist_ok=True)

src_bench = drive_results / 'benchmark_transformers_efficiency.json'
if src_bench.exists():
    shutil.copy(src_bench, 'results/benchmark_transformers_efficiency.json')
    n = len(json.load(open('results/benchmark_transformers_efficiency.json')))
    print(f'Resumindo benchmark com {n} medicoes ja salvas')
else:
    print('Comecando benchmark do zero')

for fname in [
    'best_params_tier2_n5000_gpu.json',
    'best_params_ftcur_saint_valloss.json',
]:
    src = drive_results / 'tuning' / fname
    dst = local_results / 'tuning' / fname
    if src.exists():
        shutil.copy(src, dst)
        print(f'Restaurado do Drive: {fname}')
    elif dst.exists():
        print(f'Usando arquivo do repo: {fname}')
    else:
        print(f'AVISO: tuning ausente -> {fname}')


In [ ]:
# -- Celula 7: Sync para Drive a cada 5 min ---------------------------------
%%writefile /content/sync_benchmark_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/benchmark_transformers_efficiency.json \
          "$1/benchmark_transformers_efficiency.json" 2>/dev/null
done


In [ ]:
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_benchmark_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando em background (PID {sync_proc.pid})')


In [ ]:
# -- Celula 8: Rodar benchmark real e resumivel -----------------------------
# Salva a cada medicao e retoma do JSON existente.
# Aqui a grade de N sobe para expor limite de VRAM, especialmente no SAINT,
# comparando um caso leve (ADULT) com casos mais pesados (CREDIT, HIGGS50K).
!python3 scripts/benchmark_transformers_efficiency.py \
    --datasets ADULT CREDIT HIGGS50K \
    --sizes 1000 3000 5000 10000 15000 20000 30000 48842 \
    --seed 0 \
    --output-file results/benchmark_transformers_efficiency.json


In [ ]:
# -- Celula 9: Salvar resultado final no Drive -------------------------------
import shutil, signal
from pathlib import Path

try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

src = Path('results/benchmark_transformers_efficiency.json')
dst = Path(DRIVE_PATH) / 'benchmark_transformers_efficiency.json'
if src.exists():
    shutil.copy(src, dst)
    print(f'Salvo: {dst} ({src.stat().st_size / 1024:.1f} KB)')
else:
    print('Arquivo de benchmark nao encontrado')

!ls -lh '{DRIVE_PATH}'
